# Zaid Hybrid IDS — Step 6: Five-Seed Detector Stability
This notebook measures validation stability for the Random Forest detector without Dst Port across five fixed random seeds. The existing seed-20260829 model is retained as the frozen detector; four additional models are trained only to estimate variability and are not selected or saved. The test partition is not loaded or used.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from time import perf_counter
import gc
import json
import platform
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score
)

PROJECT_DIR = Path('/content/drive/MyDrive/Zaid_Hybrid_IDS')
SPLIT_DIR = PROJECT_DIR / 'data' / 'splits'
RESULT_DIR = PROJECT_DIR / 'results'
MODEL_DIR = PROJECT_DIR / 'models'
TRAIN_PATH = SPLIT_DIR / 'train.parquet'
VALIDATION_PATH = SPLIT_DIR / 'validation.parquet'
FEATURE_MANIFEST_PATH = RESULT_DIR / 'feature_sets_manifest.json'
ABLATION_METRICS_PATH = RESULT_DIR / 'dst_port_ablation_validation_metrics.csv'
FROZEN_MODEL_PATH = MODEL_DIR / 'random_forest_no_dst_port.joblib'
SEEDS = [20260829, 20260830, 20260831, 20260832, 20260833]

for path in [TRAIN_PATH, VALIDATION_PATH, FEATURE_MANIFEST_PATH, ABLATION_METRICS_PATH, FROZEN_MODEL_PATH]:
    assert path.exists(), f'Missing required file: {path}'

print('Loading train and validation only...')
train_df = pd.read_parquet(TRAIN_PATH)
validation_df = pd.read_parquet(VALIDATION_PATH)
with open(FEATURE_MANIFEST_PATH, 'r', encoding='utf-8') as file_handle:
    feature_manifest = json.load(file_handle)
ablation_metrics = pd.read_csv(ABLATION_METRICS_PATH)
frozen_model = joblib.load(FROZEN_MODEL_PATH)

assert len(train_df) == 731315
assert len(validation_df) == 156714
print('Train:', train_df.shape)
print('Validation:', validation_df.shape)
print('Seeds:', SEEDS)
print('Test partition loaded: False')

Loading train and validation only...
Train: (731315, 81)
Validation: (156714, 81)
Seeds: [20260829, 20260830, 20260831, 20260832, 20260833]
Test partition loaded: False


In [3]:
FEATURES = feature_manifest['no_dst_port_feature_set']
assert len(FEATURES) == 60 and 'Dst Port' not in FEATURES
X_train = train_df[FEATURES].astype(np.float32)
X_validation = validation_df[FEATURES].astype(np.float32)
y_train = train_df['Label'].astype(str).ne('Benign').astype(np.int8).to_numpy()
y_validation_original = validation_df['Label'].astype(str).reset_index(drop=True)
y_validation = y_validation_original.ne('Benign').astype(np.int8).to_numpy()
assert np.isfinite(X_train.to_numpy(copy=False)).all()
assert np.isfinite(X_validation.to_numpy(copy=False)).all()

records = []
def measure(seed, probability, fit_seconds, inference_seconds):
    prediction = (np.asarray(probability) >= 0.5).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_validation, prediction, labels=[0, 1]).ravel()
    ftp_mask = y_validation_original.eq('FTP-BruteForce').to_numpy()
    ssh_mask = y_validation_original.eq('SSH-Bruteforce').to_numpy()
    return {
        'seed': int(seed), 'threshold': 0.5,
        'accuracy': accuracy_score(y_validation, prediction),
        'balanced_accuracy': balanced_accuracy_score(y_validation, prediction),
        'precision': precision_score(y_validation, prediction, zero_division=0),
        'recall': recall_score(y_validation, prediction, zero_division=0),
        'f1': f1_score(y_validation, prediction, zero_division=0),
        'macro_f1': f1_score(y_validation, prediction, average='macro'),
        'pr_auc': average_precision_score(y_validation, probability),
        'roc_auc': roc_auc_score(y_validation, probability),
        'brier_score': brier_score_loss(y_validation, probability),
        'false_positive_rate': fp / (fp + tn) if (fp + tn) else 0.0,
        'ftp_attack_recall': float(prediction[ftp_mask].mean()),
        'ssh_attack_recall': float(prediction[ssh_mask].mean()),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'fit_seconds': float(fit_seconds),
        'validation_inference_seconds': float(inference_seconds),
        'batch_mean_microseconds_per_flow': inference_seconds / len(y_validation) * 1e6
    }

def train_seed(seed):
    print(f'Training seed {seed}...')
    model = RandomForestClassifier(
        n_estimators=150, max_depth=20, min_samples_leaf=2,
        max_features='sqrt', class_weight='balanced_subsample',
        n_jobs=-1, random_state=seed
    )
    fit_start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - fit_start
    inference_start = perf_counter()
    probability = model.predict_proba(X_validation)[:, 1]
    inference_seconds = perf_counter() - inference_start
    record = measure(seed, probability, fit_seconds, inference_seconds)
    records.append(record)
    print(pd.Series(record).to_string())
    del model, probability
    gc.collect()

existing_row = ablation_metrics.loc[
    ablation_metrics['scenario'].eq('retrained_without_dst_port')
].iloc[0]
inference_start = perf_counter()
existing_probability = frozen_model.predict_proba(X_validation)[:, 1]
existing_inference_seconds = perf_counter() - inference_start
records.append(measure(
    SEEDS[0], existing_probability,
    fit_seconds=float(existing_row['fit_seconds']),
    inference_seconds=existing_inference_seconds
))
print('Frozen seed result:')
print(pd.Series(records[0]).to_string())
print('Stability evaluation setup: READY')

Frozen seed result:
seed                                2.026083e+07
threshold                           5.000000e-01
accuracy                            9.999681e-01
balanced_accuracy                   9.999674e-01
precision                           9.999475e-01
recall                              9.999650e-01
f1                                  9.999563e-01
macro_f1                            9.999656e-01
pr_auc                              1.000000e+00
roc_auc                             1.000000e+00
brier_score                         2.098592e-05
false_positive_rate                 3.012956e-05
ftp_attack_recall                   1.000000e+00
ssh_attack_recall                   9.999289e-01
tn                                  9.956700e+04
fp                                  3.000000e+00
fn                                  2.000000e+00
tp                                  5.714200e+04
fit_seconds                         1.833425e+02
validation_inference_seconds        1.207540e+00


In [4]:
train_seed(20260830)

Training seed 20260830...
seed                                2.026083e+07
threshold                           5.000000e-01
accuracy                            9.999681e-01
balanced_accuracy                   9.999674e-01
precision                           9.999475e-01
recall                              9.999650e-01
f1                                  9.999563e-01
macro_f1                            9.999656e-01
pr_auc                              1.000000e+00
roc_auc                             1.000000e+00
brier_score                         1.852452e-05
false_positive_rate                 3.012956e-05
ftp_attack_recall                   1.000000e+00
ssh_attack_recall                   9.999289e-01
tn                                  9.956700e+04
fp                                  3.000000e+00
fn                                  2.000000e+00
tp                                  5.714200e+04
fit_seconds                         1.606102e+02
validation_inference_seconds        7.28847

In [5]:
train_seed(20260831)

Training seed 20260831...
seed                                2.026083e+07
threshold                           5.000000e-01
accuracy                            9.999681e-01
balanced_accuracy                   9.999674e-01
precision                           9.999475e-01
recall                              9.999650e-01
f1                                  9.999563e-01
macro_f1                            9.999656e-01
pr_auc                              1.000000e+00
roc_auc                             1.000000e+00
brier_score                         2.055825e-05
false_positive_rate                 3.012956e-05
ftp_attack_recall                   1.000000e+00
ssh_attack_recall                   9.999289e-01
tn                                  9.956700e+04
fp                                  3.000000e+00
fn                                  2.000000e+00
tp                                  5.714200e+04
fit_seconds                         1.584386e+02
validation_inference_seconds        7.45026

In [6]:
train_seed(20260832)

Training seed 20260832...
seed                                2.026083e+07
threshold                           5.000000e-01
accuracy                            9.999681e-01
balanced_accuracy                   9.999674e-01
precision                           9.999475e-01
recall                              9.999650e-01
f1                                  9.999563e-01
macro_f1                            9.999656e-01
pr_auc                              1.000000e+00
roc_auc                             1.000000e+00
brier_score                         2.720883e-05
false_positive_rate                 3.012956e-05
ftp_attack_recall                   1.000000e+00
ssh_attack_recall                   9.999289e-01
tn                                  9.956700e+04
fp                                  3.000000e+00
fn                                  2.000000e+00
tp                                  5.714200e+04
fit_seconds                         1.540413e+02
validation_inference_seconds        7.61718

In [7]:
train_seed(20260833)

Training seed 20260833...
seed                                2.026083e+07
threshold                           5.000000e-01
accuracy                            9.999681e-01
balanced_accuracy                   9.999674e-01
precision                           9.999475e-01
recall                              9.999650e-01
f1                                  9.999563e-01
macro_f1                            9.999656e-01
pr_auc                              1.000000e+00
roc_auc                             1.000000e+00
brier_score                         2.293200e-05
false_positive_rate                 3.012956e-05
ftp_attack_recall                   1.000000e+00
ssh_attack_recall                   9.999289e-01
tn                                  9.956700e+04
fp                                  3.000000e+00
fn                                  2.000000e+00
tp                                  5.714200e+04
fit_seconds                         1.499369e+02
validation_inference_seconds        7.40222

In [8]:
assert len(records) == 5, 'Run all four additional seed cells before this cell'
seed_metrics = pd.DataFrame(records).sort_values('seed').reset_index(drop=True)
metric_columns = [
    'accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1',
    'macro_f1', 'pr_auc', 'roc_auc', 'brier_score',
    'false_positive_rate', 'ftp_attack_recall', 'ssh_attack_recall',
    'fit_seconds', 'validation_inference_seconds',
    'batch_mean_microseconds_per_flow'
]
summary = seed_metrics[metric_columns].agg(['mean', 'std', 'min', 'max']).T.reset_index()
summary = summary.rename(columns={'index': 'metric'})
macro_f1_std = float(seed_metrics['macro_f1'].std(ddof=1))
macro_f1_range = float(seed_metrics['macro_f1'].max() - seed_metrics['macro_f1'].min())
minimum_recall = float(seed_metrics['recall'].min())
stable = bool(macro_f1_std < 0.001 and minimum_recall >= 0.995)

METRICS_PATH = RESULT_DIR / 'random_forest_no_dst_port_five_seed_metrics.csv'
SUMMARY_PATH = RESULT_DIR / 'random_forest_no_dst_port_five_seed_summary.csv'
MANIFEST_PATH = RESULT_DIR / 'random_forest_no_dst_port_five_seed_manifest.json'
seed_metrics.to_csv(METRICS_PATH, index=False)
summary.to_csv(SUMMARY_PATH, index=False)
manifest = {
    'stage': 'five_seed_validation_stability',
    'seeds': SEEDS,
    'architecture': 'RandomForestClassifier',
    'feature_set': 'no_dst_port_feature_set',
    'feature_count': len(FEATURES),
    'threshold': 0.5,
    'frozen_detector_seed': SEEDS[0],
    'best_seed_selection_performed': False,
    'test_partition_loaded_or_used': False,
    'macro_f1_mean': float(seed_metrics['macro_f1'].mean()),
    'macro_f1_std': macro_f1_std,
    'macro_f1_range': macro_f1_range,
    'minimum_recall': minimum_recall,
    'stability_criterion': 'macro_f1 std < 0.001 and minimum recall >= 0.995',
    'stable': stable,
    'sklearn_version': sklearn.__version__,
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
    'python_version': sys.version,
    'platform': platform.platform()
}
with open(MANIFEST_PATH, 'w', encoding='utf-8') as file_handle:
    json.dump(manifest, file_handle, ensure_ascii=False, indent=2)

print(seed_metrics[['seed', 'macro_f1', 'f1', 'precision', 'recall', 'pr_auc', 'false_positive_rate', 'fp', 'fn']].to_string(index=False))
print('\nMacro-F1 mean:', float(seed_metrics['macro_f1'].mean()))
print('Macro-F1 std:', macro_f1_std)
print('Macro-F1 range:', macro_f1_range)
print('Minimum recall:', minimum_recall)
print('Stable:', stable)
print('Saved:', METRICS_PATH)
print('Saved:', SUMMARY_PATH)
print('Saved:', MANIFEST_PATH)
print('FIVE-SEED STABILITY PASSED: frozen detector unchanged; test remains untouched.')

    seed  macro_f1       f1  precision   recall  pr_auc  false_positive_rate  fp  fn
20260829  0.999966 0.999956   0.999948 0.999965     1.0              0.00003   3   2
20260830  0.999966 0.999956   0.999948 0.999965     1.0              0.00003   3   2
20260831  0.999966 0.999956   0.999948 0.999965     1.0              0.00003   3   2
20260832  0.999966 0.999956   0.999948 0.999965     1.0              0.00003   3   2
20260833  0.999966 0.999956   0.999948 0.999965     1.0              0.00003   3   2

Macro-F1 mean: 0.9999655715837237
Macro-F1 std: 0.0
Macro-F1 range: 0.0
Minimum recall: 0.999965000699986
Stable: True
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/random_forest_no_dst_port_five_seed_metrics.csv
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/random_forest_no_dst_port_five_seed_summary.csv
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/random_forest_no_dst_port_five_seed_manifest.json
FIVE-SEED STABILITY PASSED: frozen detector unchanged; test re